## Real Data Collection from Various Platforms

### Amazon and Flipkart Reviews

For Amazon and Flipkart, you would typically use web scraping libraries (like Beautiful Soup or Scrapy) or leverage available APIs (if permitted and accessible) to extract product reviews. Below is a placeholder for how you would collect data and integrate it with your existing `reviews` list.

In [1]:
!pip install playwright nest_asyncio
!playwright install chromium


In [2]:
from langchain_core.documents import Document
from bs4 import BeautifulSoup
import re
import concurrent.futures
import asyncio
import sys

# Existing review list
reviews = reviews if 'reviews' in locals() else []

# Simple cleaning function
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

url_sources = [
    ({"source": "amazon", "doc_type": "review"}, "https://www.amazon.in/product-reviews/B0FN7QTRPY/"),
    ({"source": "flipkart", "doc_type": "review"}, "https://www.flipkart.com/samsung-galaxy-m07-black-64-gb/product-reviews/itm0b84fbcd25ac9")
]

def fetch_with_playwright(url):
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    from langchain_community.document_loaders import PlaywrightURLLoader
    loader = PlaywrightURLLoader(urls=[url], remove_selectors=["header", "footer"])
    return loop.run_until_complete(loader.aload())

print("Loading pages with Playwright in a background thread...")
for metadata, url in url_sources:
    try:
        with concurrent.futures.ThreadPoolExecutor() as executor:
            future = executor.submit(fetch_with_playwright, url)
            web_docs = future.result()
    except Exception as e:
        import traceback
        traceback.print_exc()
        web_docs = []

    raw_texts = []
    for doc in web_docs:
        soup = BeautifulSoup(doc.page_content, "html.parser")
        text = soup.get_text(separator=" ", strip=True)
        chunks = text.split(". ")
        for chunk in chunks:
            if len(chunk.split()) > 6:
                raw_texts.append(chunk)

    raw_texts = raw_texts[:10]
    cleaned_reviews = [clean_text(review) for review in raw_texts]
    
    new_reviews = [
        Document(page_content=review, metadata=metadata.copy())
        for review in cleaned_reviews
    ]
    reviews.extend(new_reviews)
    print(f"Added {len(new_reviews)} reviews from {metadata['source']}")

print(f"Total reviews so far: {len(reviews)}")


Loading pages with Playwright in a background thread...
Added 1 reviews from amazon
Added 3 reviews from flipkart
Total reviews so far: 4


In [3]:
reviews

[Document(metadata={'source': 'amazon', 'doc_type': 'review'}, page_content='Click the button below to continue shopping Conditions of Use & Sale Privacy Notice © 1996-2025, Amazon.com, Inc'),
 Document(metadata={'source': 'flipkart', 'doc_type': 'review'}, page_content='FLIPKART Chevron Flipkart Flipkart A one-stop Shopping Selected Minutes Minutes Everything in Minutes Grocery Grocery At wholesale prices Travel Travel All your travel needs Get App Login Login Chevron New customer?Sign Up My Profile My Profile Flipkart Plus Zone Flipkart Plus Zone Orders Orders Wishlist Wishlist Become a Seller Become a Seller Rewards Rewards Gift Cards Gift Cards Notification Preferences Notification Preferences 24x7 Customer Care 24x7 Customer Care Advertise Advertise Download App Download App Login More Chevron More Become a Seller Become a Seller Notification Settings Notification Settings 24x7 Customer Care 24x7 Customer Care Advertise on Flipkart Advertise on Flipkart Cart Cart Overall Camera Ba

### YouTube Comments

For YouTube comments, you would typically use the YouTube Data API to retrieve comments from relevant product review videos. You'll need an API key for this, and you should be mindful of API quotas. You can search for videos by keywords (e.g., "phone model review") and then fetch comments for those videos.

In [4]:
import requests

response = requests.get("https://www.youtube.com")

print(response.status_code)

200


In [5]:
from youtube_transcript_api import YouTubeTranscriptApi
from deep_translator import GoogleTranslator
from langchain_core.documents import Document

youtube_reviews = []

video_ids = [
    "tmoOJIUEH9k",
    "HazHU_WkErY"
]

api = YouTubeTranscriptApi()

for video_id in video_ids:

    try:
        transcript_list = api.list(video_id)

        try:
            # Try English first
            transcript = transcript_list.find_transcript(['en'])

            fetched = transcript.fetch()

            full_text = " ".join(
                [entry.text for entry in fetched]
            )

            detected_language = "en"

        except:

            # Use first available transcript
            transcript = next(iter(transcript_list))

            fetched = transcript.fetch()

            original_text = " ".join(
                [entry.text for entry in fetched]
            )

            detected_language = transcript.language_code

            # Translate externally
            full_text = GoogleTranslator(
                source='auto',
                target='en'
            ).translate(original_text)

            print(f"Translated {detected_language} -> en")

        reviews.append(
            Document(
                page_content=full_text,
                metadata={
                    "source": "youtube",
                    "video_id": video_id,
                    "doc_type": "review",
                    "language": detected_language
                }
            )
        )

        print(f"Loaded: {video_id}")

    except Exception as e:
        print(f"Failed: {video_id}")
        print(e)

print(f"Total reviews: {len(reviews)}")

Failed: tmoOJIUEH9k
Request exception can happen due to an api connection error. Please check your connection and try again
Loaded: HazHU_WkErY
Total reviews: 5


In [26]:
reviews

[Document(metadata={'source': 'amazon', 'doc_type': 'review', 'sentiment_label': 'NEGATIVE', 'sentiment_score': 0.9569107890129089}, page_content='- Continue shopping button  \n- Conditions of Use & Sale  \n- Privacy Notice  \n- ©\u202f1996‑2025, Amazon.com, Inc.'),
 Document(metadata={'source': 'flipkart', 'doc_type': 'review', 'sentiment_label': 'POSITIVE', 'sentiment_score': 0.9822507500648499}, page_content='- Product: Black, 4\u202fGB RAM, 64\u202fGB storage.  \n- Overall ratings: 416 ratings, 24 reviews.  \n- User reviews (verified purchases):  \n  - Divyashree M (Dharmapuri), 5\u202f★, posted 2\u202fmonths ago – “Nice product.”  \n  - Unnamed reviewer, 5\u202f★ – “Great product. Loved the product! Delivery was quick and hassle‑free.”  '),
 Document(metadata={'source': 'flipkart', 'doc_type': 'review', 'sentiment_label': 'NEGATIVE', 'sentiment_score': 0.9869821667671204}, page_content='- Review 1  \n  - 5‑star rating, posted 2\u202fmonths ago  \n  - Verified purchase, 11 helpful 

### Specifications

In [6]:
mock_specs = [
    "Samsung Galaxy M07 Features: 6.7 inch HD+ display, 90Hz refresh rate, MediaTek HIC99 processor, 4GB RAM, 64GB storage (expandable to 2TB).",
    "Cameras: 50 MP main camera, 2 MP depth sensor, 8 MP front selfie camera. Battery: 5500 mAh with 25W fast charging.",
    "Connectivity: Wi-Fi, Bluetooth 5.3, 5G supported. IP54 water and dust resistance. OS: Android 15 with One UI 7.0."
]
spec_docs = [Document(page_content=spec, metadata={"source": "manufacturer", "doc_type": "spec"}) for spec in mock_specs]
reviews.extend(spec_docs)
print(f"Added {len(spec_docs)} specification documents.")
print(f"Total documents: {len(reviews)}")


Added 3 specification documents.
Total documents: 8


In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_groq import ChatGroq
import os
from getpass import getpass

# Initialize the LLM with Groq (moved here for consistent usage)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Please provide your Groq API key: ")
llm = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=os.environ["GROQ_API_KEY"])

# Define a function to format documents for the prompt, including metadata
def format_docs(docs):
    formatted_strings = []
    for doc in docs:
        content = doc.page_content
        metadata_str = ""
        if doc.metadata:
            # Include sentiment and extracted issues if present
            meta_items = []
            if 'sentiment_label' in doc.metadata:
                meta_items.append(f"Sentiment: {doc.metadata['sentiment_label']}")
            if 'extracted_issues' in doc.metadata and doc.metadata['extracted_issues']:
                issues_str = "; ".join(doc.metadata['extracted_issues'])
                meta_items.append(f"Identified Issues: {issues_str}")
            if meta_items:
                metadata_str = f" [Metadata: {', '.join(meta_items)}]\n"
        formatted_strings.append(f"{content}{metadata_str}")
    return "\n\n".join(formatted_strings)

print("LLM initialized with Groq and format_docs function defined.")

LLM initialized with Groq and format_docs function defined.


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

fluff_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data cleaner. Remove any marketing fluff, hype words, or biased language from the text. Return only the factual and critical points."),
    ("user", "{text}")
])

fluff_chain = fluff_prompt | llm | StrOutputParser()

print("Removing marketing fluff from documents...")
for doc in reviews:
    if doc.metadata.get("source") in ["manufacturer", "youtube", "amazon", "flipkart"]:
        try:
            clean_content = fluff_chain.invoke({"text": doc.page_content})
            doc.page_content = clean_content
        except Exception as e:
            print(f"Error cleaning doc: {e}")
print("Fluff removal complete.")


Removing marketing fluff from documents...
Fluff removal complete.


In [9]:
reviews

[Document(metadata={'source': 'amazon', 'doc_type': 'review'}, page_content='- Continue shopping button  \n- Conditions of Use & Sale  \n- Privacy Notice  \n- ©\u202f1996‑2025, Amazon.com, Inc.'),
 Document(metadata={'source': 'flipkart', 'doc_type': 'review'}, page_content='- Product: Black, 4\u202fGB RAM, 64\u202fGB storage.  \n- Overall ratings: 416 ratings, 24 reviews.  \n- User reviews (verified purchases):  \n  - Divyashree M (Dharmapuri), 5\u202f★, posted 2\u202fmonths ago – “Nice product.”  \n  - Unnamed reviewer, 5\u202f★ – “Great product. Loved the product! Delivery was quick and hassle‑free.”  '),
 Document(metadata={'source': 'flipkart', 'doc_type': 'review'}, page_content='- Review 1  \n  - 5‑star rating, posted 2\u202fmonths ago  \n  - Verified purchase, 11 helpful votes  \n  - Reviewer location: North Goa District (Flipkart customer)  \n  - Product details: Black color, 4\u202fGB RAM, 64\u202fGB storage, priced under\u202f8000, software updates for 6\u202fyears  \n\n- Re

After adding new data, you might want to re-run the sentiment analysis and issue extraction steps to update the metadata for the newly added documents, and then re-initialize your `splitter`, `vector_store`, and `retrievers` to include the expanded dataset.

### Important: Re-run Downstream Cells

Since the data loading methods have been updated, please **re-run all cells from the 'Sentiment Analysis and Issue Extraction' section onwards** to ensure that the `reviews` list, sentiment analysis, issue extraction, chunking, vector store, retrievers, and RAG chain are all updated with the potentially new data from the loaders.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

In [12]:
chunks = splitter.split_documents(reviews)
print(f"Split into {len(chunks)} chunks.")

Split into 11 chunks.


In [13]:
from transformers import pipeline

# Initialize a sentiment analysis pipeline
# truncation=True and max_length=512 prevent RuntimeError when input exceeds
# DistilBERT's maximum sequence length of 512 tokens
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)

# Perform sentiment analysis and update document metadata
for doc in reviews:
    # Analyze sentiment of the page_content
    result = sentiment_analyzer(doc.page_content)[0]
    sentiment_label = result['label'] # e.g., 'POSITIVE', 'NEGATIVE', 'NEUTRAL'
    sentiment_score = result['score']

    # Add sentiment to the document's metadata
    doc.metadata['sentiment_label'] = sentiment_label
    doc.metadata['sentiment_score'] = sentiment_score

print(f"Added sentiment analysis to {len(reviews)} reviews.")
# Display the first document with its new metadata for verification
print("\nFirst document with sentiment metadata:")
print(reviews[0])


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Added sentiment analysis to 8 reviews.

First document with sentiment metadata:
page_content='- Continue shopping button  
- Conditions of Use & Sale  
- Privacy Notice  
- © 1996‑2025, Amazon.com, Inc.' metadata={'source': 'amazon', 'doc_type': 'review', 'sentiment_label': 'NEGATIVE', 'sentiment_score': 0.9569107890129089}


In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field
from typing import List, Literal

# Define Pydantic model for structured issue extraction AND review classification
class ReviewAnalysis(BaseModel):
    issues: List[str] = Field(description="List of negative issues or problems identified in the phone review.")
    review_type: Literal['complaint', 'praise', 'neutral', 'mixed', 'spec'] = Field(description="The overall classification of the text.")

issue_parser = PydanticOutputParser(pydantic_object=ReviewAnalysis)

issue_extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the following text. Extract any negative issues/problems as a list. Also, classify the overall text into one of the allowed categories: complaint, praise, neutral, mixed, or spec. {format_instructions}\nText: {review}"),
    ("user", "Analyze the text."),
])

issue_extraction_chain = (
    {
        "review": RunnablePassthrough(),
        "format_instructions": lambda x: issue_parser.get_format_instructions()
    }
    | issue_extraction_prompt
    | llm
    | issue_parser
)

print("Analyzing chunks for issues and classifying review type...")
for i, chunk in enumerate(chunks):
    try:
        analysis = issue_extraction_chain.invoke(chunk.page_content)
        if analysis.issues:
            chunk.metadata['extracted_issues'] = analysis.issues
        chunk.metadata['type'] = analysis.review_type
    except Exception as e:
        print(f"Error analyzing chunk {i+1}: {e}")
        chunk.metadata['type'] = 'neutral'

print(f"Added analysis to {len(chunks)} chunks.")
if len(chunks) > 0:
    print("\nFirst chunk metadata:")
    print(chunks[0].metadata)


Analyzing chunks for issues and classifying review type...
Added analysis to 11 chunks.

First chunk metadata:
{'source': 'amazon', 'doc_type': 'review', 'type': 'neutral'}


In [15]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use a reliable, fast local embedding model instead of the Google API
# to avoid the 'len(documents) != len(embeddings)' mismatch error.
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


C:\Users\victus\AppData\Local\Temp\ipykernel_6464\2744621782.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Initialize ChromaDB and Store Embeddings

In [16]:
from langchain_community.vectorstores import FAISS

# Sanitize metadata — FAISS also requires str/int/float/bool values only
for chunk in chunks:
    for key, val in chunk.metadata.items():
        if isinstance(val, list):
            chunk.metadata[key] = '; '.join(str(v) for v in val)

vector_store = FAISS.from_documents(chunks, embedding)
print("FAISS vector store created successfully from documents.")


FAISS vector store created successfully from documents.


## Retrieval Setup

In [17]:
from langchain_community.retrievers import BM25Retriever

# Initialize BM25 retriever from documents
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4 # Retrieve top 4 documents (increased from 2)

# Initialize vector store retriever
vectorstore_retriever = vector_store.as_retriever(search_kwargs={"k": 4}) # Retrieve top 4 documents (increased from 2)

# Custom hybrid retrieval function to combine results from BM25 and VectorStore
def hybrid_retriever_invoke(query):
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = vectorstore_retriever.invoke(query)

    # Combine and de-duplicate documents
    combined_docs = {}
    for doc in bm25_docs + vector_docs:
        combined_docs[doc.page_content] = doc

    return list(combined_docs.values())

print("BM25 and VectorStore retrievers initialized. Custom hybrid retrieval function created.")

BM25 and VectorStore retrievers initialized. Custom hybrid retrieval function created.


In [18]:
# Perform a sample query using the custom hybrid retriever
query = "What are the main issues with the phone?"
retrieved_docs = hybrid_retriever_invoke(query)

print("Query:", query)
print("\nRetrieved documents:")
for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1}:\n{doc.page_content}\n---")

Query: What are the main issues with the phone?

Retrieved documents:
Document 1:
- Samsung added the Galaxy M07 to its Indian website without a formal launch event.  
- Processor: MediaTek HIC99 chipset.  
- Memory: 4 GB RAM, 64 GB internal storage, expandable via micro‑SD card up to 2 TB.  
- Operating system: Android 15 with One UI 7.0.  
- Display: 6.7‑inch HD+ panel, 90 Hz refresh rate.  
- Dimensions: 7.6 mm thickness.  
- Rear cameras: 50 MP main sensor, 2 MP depth sensor.  
- Front camera: 8 MP.  
- Battery: 5,500 mAh capacity, supports 25 W fast charging.
---
Document 2:
- Protection: IP54 rating (dust and light splash resistance).  
- Pricing in India: ₹6,999.  
- Software support: Samsung states 6 years of software updates.  
- Availability: Currently limited to the Indian market.
---
Document 3:
Cameras: 50 MP main camera, 2 MP depth sensor, 8 MP front camera. Battery: 5500 mAh, 25 W charging.
---
Document 4:
- Connectivity: Wi‑Fi, Bluetooth 5.3, 5G  
- Ingress protection: 

## Cross Encoder Reranker

In [19]:
from sentence_transformers import CrossEncoder

# Load a pre-trained Cross-Encoder model
# This model is specifically trained for re-ranking tasks
reranker_model = CrossEncoder('cross-encoder/ms-marco-TinyBERT-L-2')

print("Cross-Encoder reranker model loaded successfully.")

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-TinyBERT-L-2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-Encoder reranker model loaded successfully.


In [20]:
def rerank_documents(query, documents, model):
    # Define weights for different sources
    source_weights = {
        "amazon": 1.0,
        "flipkart": 1.0,
        "reddit": 0.9,
        "youtube": 0.5,
        "manufacturer": 0.2,
        "mock": 0.1
    }

    sentence_pairs = [[query, doc.page_content] for doc in documents]
    scores = model.predict(sentence_pairs)

    # Apply source weights
    adjusted_scores = []
    for doc, base_score in zip(documents, scores):
        source = doc.metadata.get("source", "mock")
        weight = source_weights.get(source, 1.0)
        # Adjust score by weight. Note: CrossEncoder scores are typically logits.
        # For simplicity, we just multiply the raw score here as a demonstration of weighting.
        adjusted_scores.append(base_score * weight)

    ranked_docs_with_scores = sorted(zip(documents, adjusted_scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked_docs_with_scores]

# Example query to test reranking
query = "What are the main issues with the phone?"
retrieved_docs = hybrid_retriever_invoke(query)
reranked_docs = rerank_documents(query, retrieved_docs, reranker_model)

print("Reranked documents with source weights applied:")
for i, doc in enumerate(reranked_docs[:3]):
    print(f"Document {i+1} (Source: {doc.metadata.get('source')}):\n{doc.page_content}\n---")


Reranked documents with source weights applied:
Document 1 (Source: flipkart):
- Ajay Krishnan K V, Kannur – 2.0 stars – Black, 4 GB RAM, 64 GB storage – “Very slow.”
- Flipkart Customer, Nagpur – 5.0 stars – Black, 4 GB RAM, 64 GB storage – “Awesome in this range.”
- Flipkart Customer, Shimla – 4.0 stars – Black, 4 GB RAM, 64 GB storage – “Best for budget phone.”
- Flipkart Customer, Dhule – 5.0 stars – Black, 4 GB RAM, 64 GB storage – “Good product.”
- Kamlesh Verma, Jabalpur Division – 5.0 stars – Black, 4 GB RAM, 64 GB storage – “Great product for budget purchase.”
---
Document 2 (Source: youtube):
- Samsung added the Galaxy M07 to its Indian website without a formal launch event.  
- Processor: MediaTek HIC99 chipset.  
- Memory: 4 GB RAM, 64 GB internal storage, expandable via micro‑SD card up to 2 TB.  
- Operating system: Android 15 with One UI 7.0.  
- Display: 6.7‑inch HD+ panel, 90 Hz refresh rate.  
- Dimensions: 7.6 mm thickness.  
- Rear cameras: 50 MP main sensor, 2 MP d

## LLM Integration and RAG Chain

In [21]:
from typing import Literal

class QueryIntent(BaseModel):
    intent: Literal['spec_lookup', 'review_summary', 'troubleshooting', 'general'] = Field(description="The predicted intent of the user's query.")

intent_parser = PydanticOutputParser(pydantic_object=QueryIntent)

intent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a search router. Classify the user's query intent into one of the allowed categories: spec_lookup, review_summary, troubleshooting, general.\n{format_instructions}"),
    ("user", "{query}")
])

intent_chain = (
    {
        "query": RunnablePassthrough(),
        "format_instructions": lambda x: intent_parser.get_format_instructions()
    }
    | intent_prompt
    | llm
    | intent_parser
)

sample_query = "What is the battery capacity?"
intent_result = intent_chain.invoke(sample_query)
print(f"Query: {sample_query}\nDetected Intent: {intent_result.intent}")


Query: What is the battery capacity?
Detected Intent: spec_lookup


In [22]:
from langchain_core.runnables import RunnableLambda

# Specialized retriever that only pulls documents where doc_type == 'review'
review_only_retriever = vector_store.as_retriever(search_kwargs={"k": 5, "filter": {"doc_type": "review"}})

review_summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert product reviewer. Summarize the user opinions based ONLY on the provided review documents. Do not include specifications unless mentioned in the reviews.\nReviews:\n{context}"),
    ("user", "{question}")
])

review_summarization_chain = (
    {
        "context": review_only_retriever | (lambda docs: "\n\n".join([d.page_content for d in docs])),
        "question": RunnablePassthrough()
    }
    | review_summary_prompt
    | llm
    | StrOutputParser()
)

print("Review-Only Summarization Chain initialized.")


Review-Only Summarization Chain initialized.


In [23]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

# Define Pydantic model for structured output
class PhoneReviewAnalysis(BaseModel):
    main_issues: List[str] = Field(description="List of negative issues or problems identified in the phone review.")
    good_features: List[str] = Field(description="List of positive features or strengths of the phone.")

# Set up the parser
parser = PydanticOutputParser(pydantic_object=PhoneReviewAnalysis)

# Define the prompt template with format instructions
rag_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Based on the following context, extract the phone's main issues and good features. Your output must be a JSON object with two fields: 'main_issues' (a list of distinct and concise negative aspects) and 'good_features' (a list of distinct and concise positive aspects). Use the exact JSON structure provided by the format instructions. If no information is found for a category, return an empty list for that category.\n{format_instructions}\nContext:\n{context}"),
    ("user", "{question}"),
])

# Re-create the RAG chain to use the new LLM and parser
rag_chain = (
    {
        "context": lambda x: format_docs(rerank_documents(x["question"], hybrid_retriever_invoke(x["question"]), reranker_model)),
        "question": RunnablePassthrough(),
        "format_instructions": lambda x: parser.get_format_instructions()
    }
    | rag_prompt_template
    | llm
    | parser
)

print("RAG chain re-created to use Groq LLM and PydanticOutputParser for JSON output.")

RAG chain re-created to use Groq LLM and PydanticOutputParser for JSON output.


In [24]:
# Test the RAG chain with a query
final_query = "What are the main issues reported with this phone and what are its good features?"

print(f"Final Query: {final_query}")
print("\nGenerated Answer:")
print(rag_chain.invoke({"question": final_query}))

Final Query: What are the main issues reported with this phone and what are its good features?

Generated Answer:
main_issues=['Very slow performance', 'Limited availability (India only)', 'Price may be considered high for some users', 'Only IP54 splash resistance (not full waterproof)'] good_features=['6.7‑inch HD+ display with 90\u202fHz refresh rate', 'MediaTek HIC99 processor', '4\u202fGB RAM and 64\u202fGB storage expandable to 2\u202fTB', '50\u202fMP main rear camera with 2\u202fMP depth sensor', '8\u202fMP front camera', '5,500\u202fmAh battery with 25\u202fW fast charging', 'IP54 dust and light splash resistance', 'Android\u202f15 with One UI\u202f7.0', 'Six years of software updates', '5G connectivity', 'Bluetooth\u202f5.3', 'Affordable price of ₹6,999']


## Evaluate RAG Chain Accuracy

In [25]:
import json
from typing import Dict, Any

# Define a sample test dataset with expected structured outputs
test_dataset = [
    {
        "query": "What are the main problems and advantages of this phone?",
        "expected_output": PhoneReviewAnalysis(
            main_issues=[
                "Unreliable Wi\u2011Fi connection that drops and switches to mobile data",
                "Network auto\u2011registration fails when out of coverage unless flight mode is toggled",
                "Camera and gaming performance are weak, as expected from an entry\u2011level device",
                "Battery life shorter than advertised",
                "Phone lags when opening multiple apps",
                "Battery drain after the last update",
                "Camera is terrible in low light"
            ],
            good_features=[
                "Large 6.7\u2011inch display",
                "Big battery capacity",
                "Runs Android 16 with One UI 8",
                "Handles everyday tasks (browsing, WhatsApp, etc.) well",
                "Mid\u2011range feel and build quality despite budget price",
                "Ideal for students or parents for daily use",
                "Product arrived quickly",
                "Great display and camera for a budget phone",
                "Underrated, great build quality",
                "Snappy performance for its price point",
                "Gaming is surprisingly good",
                "New UI is super smooth and responsive"
            ]
        )
    },
    {
        "query": "What are the drawbacks of the phone described in the reviews?",
        "expected_output": PhoneReviewAnalysis(
            main_issues=[
                "Dropped Wi\u2011Fi connection despite close proximity to the router",
                "Automatic switching to mobile data which can consume a lot of data",
                "Phone malfunctions from the beginning despite no physical damage",
                "Phone frequently switches off by itself",
                "Wi\u2011Fi connectivity problems",
                "Bluetooth connectivity problems",
                "Poor customer care with inadequate support",
                "Network auto\u2011registration fails when out of coverage unless flight mode is toggled",
                "Inability to handle heavy gaming (1GB+ games) causing it to hang",
                "Battery life much shorter than advertised",
                "Phone often lags when opening multiple apps",
                "Battery drain after the last update",
                "Camera is a joke / blurry photos in anything less than perfect lighting",
                "Camera is still terrible in low light",
                "Durability concerns (shattered after one drop)"
            ],
            good_features=[]
        )
    },
    {
        "query": "What are the positive aspects of the phone?",
        "expected_output": PhoneReviewAnalysis(
            main_issues=[],
            good_features=[
                "Large screen",
                "Loud speakers",
                "Long lasting battery",
                "Light weight",
                "Usable camera",
                "Good entry level phone",
                "Large 6.7 in display",
                "Android 16 with One UI 8",
                "Great for normal use",
                "Quality and feel of a midrange product",
                "Does all essentials well (Internet browsing, WhatsApp)",
                "Good for students daily use",
                "Good for parents who are starting to use smartphones",
                "Product arrived quickly",
                "Great display",
                "Highly recommend for casual users",
                "Great build quality",
                "Snappy performance for its price point",
                "Gaming is surprisingly good",
                "New UI is super smooth and responsive"
            ]
        )
    }
]

def calculate_f1_score(predicted_list: List[str], gold_list: List[str]) -> float:
    if not gold_list and not predicted_list:
        return 1.0 # Perfect score if both are empty
    if not gold_list or not predicted_list:
        return 0.0 # Zero score if one is empty but not the other

    # Normalize and lower case for case-insensitive comparison
    predicted_set = set(item.lower().strip() for item in predicted_list)
    gold_set = set(item.lower().strip() for item in gold_list)

    true_positives = len(predicted_set.intersection(gold_set))
    false_positives = len(predicted_set - gold_set)
    false_negatives = len(gold_set - predicted_set)

    if (true_positives + false_positives) == 0:
        precision = 0.0
    else:
        precision = true_positives / (true_positives + false_positives)

    if (true_positives + false_negatives) == 0:
        recall = 0.0
    else:
        recall = true_positives / (true_positives + false_negatives)

    if (precision + recall) == 0:
        f1 = 0.0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)
    return f1

def evaluate_rag_chain(dataset: List[Dict[str, Any]], chain) -> Dict[str, Any]:
    total_issue_f1 = 0.0
    total_feature_f1 = 0.0
    num_tests = len(dataset)

    print(f"\n--- Evaluating RAG Chain on {num_tests} Samples ---\n")

    for i, test_case in enumerate(dataset):
        query = test_case["query"]
        expected_output = test_case["expected_output"]

        print(f"Test Case {i+1}:")
        print(f"  Query: {query}")

        try:
            generated_output = chain.invoke({"question": query})
            print(f"  Generated Issues: {generated_output.main_issues}")
            print(f"  Expected Issues: {expected_output.main_issues}")
            print(f"  Generated Features: {generated_output.good_features}")
            print(f"  Expected Features: {expected_output.good_features}")

            issue_f1 = calculate_f1_score(generated_output.main_issues, expected_output.main_issues)
            feature_f1 = calculate_f1_score(generated_output.good_features, expected_output.good_features)

            total_issue_f1 += issue_f1
            total_feature_f1 += feature_f1

            print(f"  Issue F1 Score: {issue_f1:.4f}")
            print(f"  Feature F1 Score: {feature_f1:.4f}")
            print("----------------------------------------")

        except Exception as e:
            print(f"  Error during chain invocation: {e}")
            print("----------------------------------------")
            num_tests -= 1 # Reduce count for tests that errored out

    if num_tests == 0:
        return {"average_issue_f1": 0.0, "average_feature_f1": 0.0}

    average_issue_f1 = total_issue_f1 / num_tests
    average_feature_f1 = total_feature_f1 / num_tests

    print(f"\n--- Evaluation Summary ---")
    print(f"Average Issue F1 Score: {average_issue_f1:.4f}")
    print(f"Average Feature F1 Score: {average_feature_f1:.4f}")

    return {
        "average_issue_f1": average_issue_f1,
        "average_feature_f1": average_feature_f1,
    }

# Update test_dataset with all collected reviews if necessary
# Assuming test_dataset is already defined in a previous cell

# Run the evaluation
evaluation_results = evaluate_rag_chain(test_dataset, rag_chain)
print(f"\nEvaluation Results: {json.dumps(evaluation_results, indent=2)}")


--- Evaluating RAG Chain on 3 Samples ---

Test Case 1:
  Query: What are the main problems and advantages of this phone?
  Generated Issues: ['Slow performance', 'Limited availability (only in India)', 'Price may be considered high for a budget phone', 'Only IP54 water resistance (not fully waterproof)']
  Expected Issues: ['Unreliable Wi‑Fi connection that drops and switches to mobile data', 'Network auto‑registration fails when out of coverage unless flight mode is toggled', 'Camera and gaming performance are weak, as expected from an entry‑level device', 'Battery life shorter than advertised', 'Phone lags when opening multiple apps', 'Battery drain after the last update', 'Camera is terrible in low light']
  Generated Features: ['Good value for budget segment', 'Large 5,500\u202fmAh battery with 25\u202fW fast charging', '90\u202fHz HD+ display', '50\u202fMP main camera with depth sensor', 'Expandable storage up to 2\u202fTB', '5G connectivity', 'IP54 dust and splash resistance', 